## Creating a chatbot tool capabilities from arxiv , wikipedia search and some custom functions

In [9]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
%pip install arxiv
%pip install wikipedia

Note: you may need to restart the kernel to use updated packages.
  Using cached wikipedia-1.4.0.tar.gz (27 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11758 sha256=eac4c2722ec56ef221eb5ad4ee03db4750198d9e55a7a0d8db30ff820448328d
  Stored in directory: /Users/kaushik003/Library/Caches/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [wikipedia]
Note: you may need to restart the kernel to use updated packages.


In [5]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)


arxiv


In [7]:
arxiv.invoke("attention is all you need")

'Published: 2021-05-06\nTitle: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet\nAuthors: Luke Melas-Kyriazi\nSummary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifi'

In [10]:
api_wrapper_wikipedia = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
wikipedia = WikipediaQueryRun(api_wrapper=api_wrapper_wikipedia)
print(wikipedia.name)

wikipedia


In [11]:
wikipedia.invoke("attention is all you need")

'Page: Attention Is All You Need\nSummary: "Attention Is All You Need" is a 2017 research paper in machine learning authored by eight scientists working at Google. The paper introduced a new deep learning architecture known as the transformer, based on the attention mechanism proposed in 2014 by Bahdanau et al. The transformer approach it describes has become the main architecture of a wide variety of AI, such as large language models. At the time, the focus of the research was on improving Seq2se'

In [22]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [23]:
%pip install langchain-tavily
from langchain_tavily import TavilySearch

tavily = TavilySearch()

Note: you may need to restart the kernel to use updated packages.


In [25]:
tavily.invoke("provide me recent ai news")

{'query': 'provide me recent ai news',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.youtube.com/@AINewsOfficial',
   'title': 'AI News - YouTube',
   'content': 'AI News delivers the latest breakthroughs in artificial intelligence, machine learning, deep learning, brain computer interface, robotics, and other future',
   'score': 0.9980882,
   'raw_content': None},
  {'url': 'https://www.artificialintelligence-news.com/',
   'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide.',
   'score': 0.99722075,
   'raw_content': None},
  {'url': 'https://www.nbcnews.com/artificial-intelligence',
   'title': 'Artificial intelligence - NBC News',
   'content': "The latest news and top stories on artificial intelligence, including AI chatbots like Microsoft's ChatGPT,

In [26]:
tools = [arxiv, wikipedia, tavily]

In [32]:
%pip install langchain-groq
from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen/qwen3-32b")
llm_with_tools = llm.bind_tools(tools)


Note: you may need to restart the kernel to use updated packages.


In [33]:
from pprint import pprint
from langchain_core.messages import HumanMessage, AIMessage
llm_with_tools.invoke([HumanMessage(content="What are the recent advancements in AI?")])

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about recent advancements in AI. Let me break this down.\n\nFirst, "recent" implies that I need to look for information from a specific time frame. Since the user didn\'t mention a specific date, maybe the last few months. But I should check the parameters of the tools available. The Tavily search tool has parameters like start_date and time_range. \n\nLooking at the Tavily function, the time_range can be set to "month" or "year". Since the user is asking for recent advancements, setting time_range to "month" would get the latest developments. Alternatively, using start_date with a specific date might be more precise, but if the user hasn\'t specified exact dates, using time_range is safer.\n\nAlso, the search_depth parameter can be set to "advanced" to ensure comprehensive results, as advancements in AI are a complex topic. The include_images parameter could be set to true if the user might benefit

In [36]:
from pydantic import BaseModel
from typing import TypedDict, Annotated, Sequence
from langgraph.graph import StateGraph, END, add_messages
from langchain_core.messages import AnyMessage
class State(BaseModel):
    messages: Annotated[list[AnyMessage], add_messages]

In [44]:
## chatbot
from IPython.display import display, Image
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

##node def
def tool_calls(state: State):
    return {"messages": [llm_with_tools.invoke(state.messages)]}

graph = StateGraph(State)
graph.add_node("tool_calls", tool_calls)
graph.add_node("tools", ToolNode(tools=tools))

graph.add_edge(START, "tool_calls")
graph.add_conditional_edges(
    "tool_calls", 
    tools_condition,
 )
graph.add_edge("tools", END)

app = graph.compile()

In [46]:
messages = app.invoke({"messages": [HumanMessage(content="what is attention is all you need?")]})
for m in messages["messages"]:
    m.pretty_print()

================================ Human Message =================================

what is attention is all you need?
================================== Ai Message ==================================
Tool Calls:
  arxiv (x6k2skyg4)
 Call ID: x6k2skyg4
  Args:
    query: Attention is All You Need
================================= Tool Message =================================
Name: arxiv

Published: 2021-05-06
Title: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet
Authors: Luke Melas-Kyriazi
Summary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifi
